# Phishing Email Detection Using Machine Learning and Deep Learning

## Introduction

Phishing emails are one of the most common cybersecurity threats and are designed to deceive users into revealing sensitive information, downloading malicious files, or visiting fraudulent websites.

The objective of this experiment is to develop and compare machine learning and deep learning approaches for identifying malicious or spam/phishing emails based primarily on their textual content.

Two main approaches are investigated:

1. **Traditional Machine Learning**
   - TF-IDF text representation
   - Logistic Regression
   - Random Forest

2. **Deep Learning**
   - Word embeddings learned during training
   - Bidirectional Long Short-Term Memory (BiLSTM) neural network implemented using PyTorch

The models are evaluated using:

- Precision
- Recall
- F1-score
- Confusion Matrix
- Receiver Operating Characteristic (ROC) Curve
- Area Under the ROC Curve (AUC)

Recall is particularly important in phishing detection because a false negative represents a malicious email that has incorrectly been classified as legitimate.

In [0]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

# Basic data manipulation
import pandas as pd
import numpy as np

# Text processing
import re
import string
from collections import Counter

# Visualisation
import matplotlib.pyplot as plt

# NLTK preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    ConfusionMatrixDisplay
)

# PyTorch
import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Miscellaneous
import random
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print(
    "Device:",
    "CUDA GPU" if torch.cuda.is_available() else "CPU"
)

In [0]:
# ============================================================
# DOWNLOAD NLTK RESOURCES
# ============================================================

# English stopwords such as "the", "is", "and", etc.
nltk.download("stopwords")

# WordNet database is required for lemmatization.
nltk.download("wordnet")

# Additional WordNet language data.
nltk.download("omw-1.4")

print("NLTK resources downloaded successfully.")

## Dataset Loading

The selected dataset contains email subject lines, message bodies and a classification label indicating whether an email is legitimate or malicious/spam.

The subject and body of the email will later be combined so that the models can learn from information appearing in both locations.

In [0]:
# ============================================================
# LOAD DATASET
# ============================================================

# Change this path to the location of your CSV file.
FILE_PATH = "datasets/enron_spam_data.csv"

df = pd.read_csv(FILE_PATH)

# Display the first five observations.
display(df.head())

print("\nDataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

In [0]:
df.head()

In [0]:
# ============================================================
# REMOVE UNNECESSARY COLUMNS
# ============================================================

# "Unnamed: 0" is normally generated when a DataFrame index
# is accidentally written to a CSV file.
df.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
    inplace=True
)

print(df.columns.tolist())

## Data Quality Assessment

Before text preprocessing, the dataset is examined for:

- Missing values
- Duplicate observations
- Invalid labels
- Incorrect data types

Missing subject or message values can safely be replaced with empty strings because the remaining email component may still contain useful information.

In [0]:
# ============================================================
# CHECK MISSING VALUES
# ============================================================

missing_values = df.isnull().sum()

print("Missing values:\n")
print(missing_values)

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())

In [0]:
# ============================================================
# HANDLE MISSING VALUES
# ============================================================

# Replace missing email subjects with an empty string.
df["Subject"] = df["Subject"].fillna("")

# Replace missing message bodies with an empty string.
df["Message"] = df["Message"].fillna("")

# Remove exact duplicate records.
df = df.drop_duplicates().reset_index(drop=True)

# Convert Date into pandas datetime format.
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

print("Dataset shape after cleaning:", df.shape)

In [0]:
# ============================================================
# INSPECT TARGET LABELS
# ============================================================

print("Unique target labels:")
print(df["Spam/Ham"].unique())

print("\nClass counts:")
print(df["Spam/Ham"].value_counts())

## Target Encoding

Machine learning algorithms require numerical target variables.

For the current dataset:

- `ham` = 0 → legitimate email
- `spam` = 1 → suspicious/spam email

For a dedicated phishing dataset, the same approach can be used with:

- legitimate = 0
- phishing = 1

In [0]:
# ============================================================
# STANDARDISE TARGET LABELS
# ============================================================

# Convert labels to lowercase and remove unnecessary whitespace.
df["Spam/Ham"] = (
    df["Spam/Ham"]
    .astype(str)
    .str.lower()
    .str.strip()
)


# Mapping supports several common email dataset label names.
label_mapping = {
    "ham": 0,
    "legitimate": 0,
    "safe": 0,
    "normal": 0,
    "non-phishing": 0,

    "spam": 1,
    "phishing": 1,
    "phish": 1,
    "malicious": 1
}


df["Label"] = df["Spam/Ham"].map(label_mapping)


# Check whether any labels could not be mapped.
if df["Label"].isnull().any():
    print("Unrecognised labels:")
    print(
        df.loc[
            df["Label"].isnull(),
            "Spam/Ham"
        ].unique()
    )
else:
    print("All labels successfully encoded.")


# Convert label to integer.
df["Label"] = df["Label"].astype(int)

print("\nEncoded class distribution:")
print(df["Label"].value_counts())

In [0]:
# ============================================================
# COMBINE SUBJECT AND MESSAGE
# ============================================================

# Subject lines often contain highly useful phishing indicators.
# Therefore, the subject and email body are combined.

df["Raw_Text"] = (
    df["Subject"].astype(str)
    + " "
    + df["Message"].astype(str)
)

display(
    df[
        [
            "Subject",
            "Message",
            "Raw_Text",
            "Label"
        ]
    ].head()
)

## Exploratory Data Analysis

The first visualisation examines the class distribution.

A strongly imbalanced dataset may cause a classifier to favour the majority class. This is particularly problematic in cybersecurity because phishing emails may represent the minority class while still being the most important emails to detect.

In [0]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================

class_counts = df["Label"].value_counts().sort_index()

class_names = [
    "Legitimate / Ham",
    "Spam / Phishing"
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    class_names,
    class_counts.values
)

plt.title(
    "Distribution of Email Classes",
    fontsize=14
)

plt.ylabel("Number of Emails")
plt.xlabel("Email Class")

# Add exact counts above the bars.
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# EMAIL LENGTH ANALYSIS
# ============================================================

# Number of characters.
df["Character_Count"] = df["Raw_Text"].str.len()

# Number of words.
df["Word_Count"] = (
    df["Raw_Text"]
    .str.split()
    .str.len()
)


print(
    df.groupby("Label")[
        ["Character_Count", "Word_Count"]
    ].describe()
)

In [0]:
# ============================================================
# WORD COUNT DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

# Use the 99th percentile to prevent a few extremely long emails
# from making the histogram difficult to interpret.
max_length = df["Word_Count"].quantile(0.99)

plt.hist(
    df.loc[df["Label"] == 0, "Word_Count"],
    bins=50,
    alpha=0.6,
    label="Legitimate / Ham"
)

plt.hist(
    df.loc[df["Label"] == 1, "Word_Count"],
    bins=50,
    alpha=0.6,
    label="Spam / Phishing"
)

plt.xlim(0, max_length)

plt.title("Distribution of Email Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.legend()
plt.tight_layout()
plt.show()

The email-length distribution can indicate whether malicious emails have noticeably different structural characteristics from legitimate emails.

However, message length alone is unlikely to provide sufficient discriminatory information. Consequently, the actual textual content of each email is analysed using NLP techniques.

## Text Preprocessing

Email text contains substantial noise including punctuation, inconsistent capitalisation and common words that provide little discriminatory value.

The preprocessing pipeline performs the following operations:

1. Convert text to lowercase.
2. Replace web links with the token `urltoken`.
3. Replace email addresses with the token `emailtoken`.
4. Remove HTML tags.
5. Remove punctuation and non-alphabetic symbols.
6. Remove English stopwords.
7. Apply WordNet lemmatization.

URLs and email addresses are replaced rather than completely removed because their presence may itself provide useful information for phishing detection.

In [0]:
# ============================================================
# INITIALISE NLP COMPONENTS
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """
    Clean and preprocess an individual email.

    Steps:
    1. Lowercase
    2. Preserve URL information as a special token
    3. Preserve email-address information as a special token
    4. Remove HTML
    5. Remove punctuation/numbers
    6. Remove stopwords
    7. Lemmatize words
    """

    # Make sure the input is a string.
    text = str(text)

    # Convert to lowercase.
    text = text.lower()

    # Replace URLs with a meaningful placeholder.
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " urltoken ",
        text
    )

    # Replace email addresses with a placeholder.
    text = re.sub(
        r"\S+@\S+",
        " emailtoken ",
        text
    )

    # Remove HTML tags.
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Keep alphabetic characters only.
    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    # Replace repeated whitespace with one space.
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Split text into individual tokens.
    words = text.split()

    # Remove stopwords and very short words.
    words = [
        word
        for word in words
        if word not in stop_words
        and len(word) > 1
    ]

    # Lemmatize each remaining word.
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Reconstruct cleaned email.
    return " ".join(words)

In [0]:
# ============================================================
# APPLY TEXT PREPROCESSING
# ============================================================

df["Clean_Text"] = df["Raw_Text"].apply(
    preprocess_text
)

print("Original email:\n")
print(df.loc[0, "Raw_Text"][:1000])

print("\n" + "=" * 80)

print("\nPreprocessed email:\n")
print(df.loc[0, "Clean_Text"][:1000])

In [0]:
display(
    df[
        [
            "Raw_Text",
            "Clean_Text",
            "Spam/Ham",
            "Label"
        ]
    ].head()
)

In [0]:
# ============================================================
# FUNCTION TO VISUALISE MOST COMMON WORDS
# ============================================================

def plot_common_words(text_series, title, top_n=20):

    # Join every document together.
    all_words = " ".join(
        text_series.astype(str)
    ).split()

    # Count word occurrences.
    word_counts = Counter(all_words)

    # Obtain top N words.
    common_words = word_counts.most_common(top_n)

    words = [
        item[0]
        for item in common_words
    ]

    counts = [
        item[1]
        for item in common_words
    ]

    plt.figure(figsize=(10, 6))

    plt.barh(
        words[::-1],
        counts[::-1]
    )

    plt.title(title)
    plt.xlabel("Frequency")

    plt.tight_layout()
    plt.show()

In [0]:
# Most common words in legitimate emails.
plot_common_words(
    df.loc[
        df["Label"] == 0,
        "Clean_Text"
    ],
    "Most Frequent Words in Legitimate Emails"
)

In [0]:
# Most common words in spam/phishing emails.
plot_common_words(
    df.loc[
        df["Label"] == 1,
        "Clean_Text"
    ],
    "Most Frequent Words in Spam / Phishing Emails"
)

##17. Train / Validation / Test Split

## Dataset Splitting

The dataset is divided into three subsets:

- **70% Training data**: used to train model parameters.
- **15% Validation data**: used to monitor the deep-learning model during training.
- **15% Test data**: used only for final model evaluation.

Stratified sampling is used so that approximately the same phishing/legitimate class proportions are maintained in each subset.

Importantly, TF-IDF and vocabulary construction are performed only after the split. This prevents information from the test set leaking into the training process.

In [0]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X = df["Clean_Text"]
y = df["Label"]


# First create:
# 70% training
# 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)


# Split temporary data equally:
# 15% validation
# 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)


print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

# Traditional Machine Learning Approach

## TF-IDF Feature Extraction

Machine learning algorithms cannot directly process raw textual documents. Therefore, the emails are transformed into numerical feature vectors.

Term Frequency-Inverse Document Frequency (TF-IDF) gives relatively high weights to words that are important within a particular document while reducing the influence of words that occur frequently across the entire corpus.

The vectorizer uses both:

- **Unigrams**: individual words
- **Bigrams**: two-word sequences

Using bigrams may help identify phrases such as "click here", "verify account", or "urgent action".

In [0]:
# ============================================================
# TF-IDF FEATURE EXTRACTION
# ============================================================

tfidf = TfidfVectorizer(
    # Use single words and two-word combinations.
    ngram_range=(1, 2),

    # Restrict dimensionality.
    max_features=15000,

    # Ignore extremely rare terms.
    min_df=2,

    # Apply logarithmic term-frequency scaling.
    sublinear_tf=True
)


# IMPORTANT:
# Fit TF-IDF using training data ONLY.
X_train_tfidf = tfidf.fit_transform(X_train)

# Validation and test data use the already-fitted vectorizer.
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)


print(
    "Training TF-IDF matrix:",
    X_train_tfidf.shape
)

print(
    "Test TF-IDF matrix:",
    X_test_tfidf.shape
)

In [0]:
# ============================================================
# VISUALISE A SMALL TF-IDF MATRIX
# ============================================================

# Get all words/features learned by TF-IDF.
feature_names = np.array(
    tfidf.get_feature_names_out()
)


# ------------------------------------------------------------
# Select a small number of emails for visualisation
# ------------------------------------------------------------

NUM_EMAILS = 8
NUM_TERMS = 15


# Take the first few training emails.
sample_matrix = X_train_tfidf[:NUM_EMAILS]


# ------------------------------------------------------------
# Find the most important terms across these sample emails
# ------------------------------------------------------------

# Calculate total TF-IDF score for every term.
term_scores = np.asarray(
    sample_matrix.sum(axis=0)
).flatten()


# Get indices of the highest-scoring terms.
top_term_indices = np.argsort(
    term_scores
)[-NUM_TERMS:]


# Extract only those important terms.
top_terms = feature_names[
    top_term_indices
]


# Extract TF-IDF values for those terms.
heatmap_data = (
    sample_matrix[
        :,
        top_term_indices
    ]
    .toarray()
)


# Convert into a DataFrame for easier interpretation.
tfidf_heatmap_df = pd.DataFrame(
    heatmap_data,
    columns=top_terms,
    index=[
        f"Email {i+1}"
        for i in range(NUM_EMAILS)
    ]
)


display(tfidf_heatmap_df.round(3))

## Logistic Regression

Logistic Regression is an effective baseline classifier for high-dimensional sparse text data such as TF-IDF vectors.

`class_weight="balanced"` is used so that the algorithm gives more importance to minority-class examples if the dataset is imbalanced.

In [0]:
# ============================================================
# LOGISTIC REGRESSION
# ============================================================

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED
)


# Train the classifier.
logistic_model.fit(
    X_train_tfidf,
    y_train
)


# Generate predicted classes.
lr_predictions = logistic_model.predict(
    X_test_tfidf
)


# Generate phishing/spam probabilities.
lr_probabilities = logistic_model.predict_proba(
    X_test_tfidf
)[:, 1]


print("Logistic Regression training completed.")

In [0]:
# ============================================================
# LOGISTIC REGRESSION CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_test,
        lr_predictions,
        target_names=[
            "Legitimate",
            "Spam/Phishing"
        ],
        digits=4
    )
)

In [0]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    lr_predictions,
    display_labels=[
        "Legitimate",
        "Spam/Phishing"
    ],
    cmap="Blues",
    values_format="d",
    ax=ax
)

plt.title(
    "Logistic Regression Confusion Matrix"
)

plt.tight_layout()
plt.show()

The confusion matrix provides additional information beyond overall accuracy:

- **True Negative (TN):** legitimate email correctly detected.
- **False Positive (FP):** legitimate email incorrectly marked as malicious.
- **False Negative (FN):** malicious email incorrectly classified as legitimate.
- **True Positive (TP):** malicious email correctly detected.

For cybersecurity applications, false negatives are particularly important because they correspond to threats that successfully bypass the detection model.

## Random Forest Classifier

A Random Forest combines multiple decision trees and makes a final classification based on their aggregated predictions.

Unlike Logistic Regression, a Random Forest can capture nonlinear relationships between features. It is included as a second traditional machine-learning benchmark.

In [0]:
# ============================================================
# RANDOM FOREST
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)


# Train model.
rf_model.fit(
    X_train_tfidf,
    y_train
)


# Predicted labels.
rf_predictions = rf_model.predict(
    X_test_tfidf
)


# Probabilities required for ROC analysis.
rf_probabilities = rf_model.predict_proba(
    X_test_tfidf
)[:, 1]


print("Random Forest training completed.")

In [0]:
print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=[
            "Legitimate",
            "Spam/Phishing"
        ],
        digits=4
    )
)

In [0]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_predictions,
    display_labels=[
        "Legitimate",
        "Spam/Phishing"
    ],
    cmap="Blues",
    values_format="d",
    ax=ax
)

plt.title(
    "Random Forest Confusion Matrix"
)

plt.tight_layout()
plt.show()

## Interpretability of the Logistic Regression Model

One advantage of Logistic Regression is that the coefficients can be inspected to determine which textual features contributed most strongly toward each class.

A large positive coefficient indicates a term associated with the malicious/spam class, while a large negative coefficient indicates a term associated with legitimate emails.

In [0]:
# ============================================================
# EXTRACT LOGISTIC REGRESSION FEATURE IMPORTANCE
# ============================================================

feature_names = np.array(
    tfidf.get_feature_names_out()
)

coefficients = logistic_model.coef_[0]


# Obtain indices for strongest positive coefficients.
spam_indices = np.argsort(
    coefficients
)[-20:]


# Obtain indices for strongest negative coefficients.
ham_indices = np.argsort(
    coefficients
)[:20]


spam_words = feature_names[spam_indices]
spam_scores = coefficients[spam_indices]

ham_words = feature_names[ham_indices]
ham_scores = coefficients[ham_indices]

In [0]:
# ============================================================
# MOST IMPORTANT SPAM / PHISHING FEATURES
# ============================================================

plt.figure(figsize=(10, 7))

plt.barh(
    spam_words,
    spam_scores
)

plt.title(
    "Top TF-IDF Features Associated with Spam / Phishing"
)

plt.xlabel(
    "Logistic Regression Coefficient"
)

plt.tight_layout()
plt.show()

# Deep Learning Approach: Bidirectional LSTM

Traditional TF-IDF methods treat emails largely as unordered collections of terms. Deep-learning sequence models can additionally capture relationships between words appearing at different positions in an email.

A Bidirectional Long Short-Term Memory (BiLSTM) network is therefore implemented using PyTorch.

The architecture contains:

1. **Embedding Layer**
   - Converts each word index into a dense vector.
   - Embeddings are learned automatically during training.

2. **Bidirectional LSTM**
   - Processes the email from left-to-right and right-to-left.
   - Allows the classifier to learn contextual dependencies.

3. **Dropout**
   - Reduces overfitting.

4. **Fully Connected Layer**
   - Generates the final binary classification score.

5. **Sigmoid interpretation**
   - Converts the output logit into a probability between 0 and 1.